# DATASCAPE 2026 — ConvNeXt-Tiny OOF Relabeling (320 px, Effective Batch 128)

Notebook Colab ini hanya memperbaiki **training manifest**. Tidak membaca pixel test, tidak membuat crop, dan tidak membuat submission.

Pipeline:

1. membaca manifest V1 yang sudah bebas exact duplicate dan file rusak;
2. fine-tune ConvNeXt-Tiny pretrained secara out-of-fold pada data train resmi;
3. memakai input 320×320 dan target effective batch 128 dengan fallback VRAM otomatis;
4. membuat probabilitas OOF untuk setiap gambar;
5. relabel otomatis hanya untuk kandidat ber-confidence sangat tinggi;
6. menyelesaikan atau mengarantina grup near-duplicate secara otomatis;
7. membuat CSV antrean review dan contact sheet untuk audit visual;
8. mengekspor manifest final untuk student training.

Aktifkan **T4 GPU**, lalu jalankan **Run all**.


In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


## Kepatuhan aturan dan AI Declaration

OpenAI Codex digunakan untuk brainstorming, coding, debugging, dan dokumentasi. ConvNeXt-Tiny pretrained di-fine-tune hanya pada data train resmi dengan input 320×320 dan effective batch 128. Semua perubahan label berasal dari prediksi OOF; tidak ada manual labeling.

Raw image tidak diubah. Output utama adalah manifest yang menyimpan label awal, label akhir, probabilitas, margin, fold, dan alasan keputusan.


In [ ]:
import importlib.util
import hashlib
import io
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn',
    'torch': 'torch', 'torchvision': 'torchvision', 'tqdm': 'tqdm',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont, ImageOps
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB, '| CUDA:', torch.cuda.is_available(), '| PyTorch:', torch.__version__)

## 1. Konfigurasi relabeling

Threshold dibuat konservatif. Relabel hanya terjadi jika teacher OOF sangat yakin, label lama hampir tidak didukung, dan margin dua kelas teratas besar. Perubahan per kelas asal dibatasi agar teacher tidak mengubah distribusi secara massal.

Target batch 128 adalah **effective batch**. Notebook mencoba physical batch 128 lebih dahulu, lalu turun ke 64, 32, atau 16 saat VRAM tidak cukup dan menggantinya dengan gradient accumulation. Dengan begitu optimisasi tetap melihat target batch yang sama tanpa membuat T4 kehabisan memori.


In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
INDEX_TO_CLASS = {index: label for label, index in CLASS_TO_INDEX.items()}
COMPETITION = 'datascape-2026-data-mining-competition'
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

DATA_SOURCE = 'auto'
LOCAL_DATA_PATH = ''
PREPROCESSING_V1_SOURCE = ''

USE_GOOGLE_DRIVE = True
OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/relabel_convnext_tiny_320')
LOCAL_OUTPUT_DIR = Path('artifacts/relabel_convnext_tiny_320')
EXTRACT_DIR = Path('/content/datascape_dataset')
PREPROCESS_EXTRACT_DIR = Path('/content/datascape_preprocessing_v1')

TEACHER_MODEL = 'convnext_tiny'
TEACHER_IMAGE_SIZE = 320
TEACHER_TARGET_BATCH_SIZE = 128
TEACHER_BATCH_CANDIDATES = (128, 64, 32, 16)
TEACHER_BATCH_SIZE = None  # dipilih lewat probe VRAM
TEACHER_GRAD_ACCUM_STEPS = None
TEACHER_EPOCHS = 15
TEACHER_PATIENCE = 4
TEACHER_LR = 1e-4
TEACHER_WEIGHT_DECAY = 0.05
TEACHER_LABEL_SMOOTHING = 0.05
TEACHER_MAX_GRAD_NORM = 1.0
FOLDS = 5
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 2) - 1))

PSEUDO_MIN_ALTERNATIVE_PROB = 0.98
PSEUDO_MAX_ORIGINAL_PROB = 0.03
PSEUDO_MIN_MARGIN = 0.85
MAX_RELABEL_FRACTION_PER_ORIGINAL_CLASS = 0.015

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read Kaggle credentials from Colab Secrets or the environment.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
print('Output:', OUTPUT_DIR)


## 2. Hubungkan dataset resmi dan manifest V1

Manifest V1 menyediakan path relatif, label folder, exact-hash, grup near-duplicate, fold, dan flag kualitas. Notebook ini hanya membuka gambar yang tercantum sebagai training.


In [ ]:
def get_kaggle_token():
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError('Set KAGGLE_API_TOKEN di Colab Secrets atau environment.')
    return token


def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_root(start, filename, directories=()):
    start = Path(start).expanduser()
    candidates = [start] if start.is_dir() else []
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob(filename))
    for candidate in candidates:
        if (candidate / filename).is_file() and all((candidate / d).is_dir() for d in directories):
            return candidate.resolve()
    raise FileNotFoundError(f'{filename} tidak ditemukan di {start}')


def resolve_v1_root():
    if PREPROCESSING_V1_SOURCE:
        source = Path(PREPROCESSING_V1_SOURCE).expanduser()
        if source.suffix.lower() == '.zip':
            source = safe_extract(source, PREPROCESS_EXTRACT_DIR)
        return find_root(source, 'training_manifest.csv')
    for candidate in [Path('/content/datascape_preprocessing'), Path('/content/artifacts/preprocessing_colab'),
                      Path.cwd() / 'artifacts' / 'preprocessing_colab']:
        if (candidate / 'training_manifest.csv').is_file():
            return candidate.resolve()
    if IN_COLAB:
        archives = sorted(Path('/content').glob('*preprocess*.zip'))
        if archives:
            return find_root(safe_extract(archives[0], PREPROCESS_EXTRACT_DIR), 'training_manifest.csv')
    raise FileNotFoundError('Hasil preprocessing V1 tidak ditemukan.')


def find_training_root(start):
    start = Path(start).expanduser().resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob('Train_Set') if path.is_dir())
    for candidate in candidates:
        if all((candidate / 'Train_Set' / label).is_dir() for label in CLASSES):
            return candidate
    raise FileNotFoundError(f'Root Train_Set tidak ditemukan di {start}')


def resolve_dataset_root():
    if DATA_SOURCE in {'auto', 'folder'} and LOCAL_DATA_PATH:
        path = Path(LOCAL_DATA_PATH).expanduser()
        if path.exists():
            return find_training_root(path)
    for candidate in [Path.cwd() / COMPETITION, Path('/content') / COMPETITION]:
        if DATA_SOURCE == 'auto' and candidate.exists():
            return find_training_root(candidate)
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(get_kaggle_token())
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_training_root(safe_extract(downloaded, EXTRACT_DIR))
    return find_training_root(downloaded)


V1_ROOT = resolve_v1_root()
DATA_ROOT = resolve_dataset_root()
manifest = pd.read_csv(V1_ROOT / 'training_manifest.csv')
near_pairs = pd.read_csv(V1_ROOT / 'near_duplicate_candidates.csv')
manifest['source_path'] = manifest.filepath.map(lambda value: str(DATA_ROOT / Path(value)))
manifest['label_original'] = manifest.label
assert manifest.source_path.map(lambda value: Path(value).is_file()).all()
assert manifest.groupby('split_group').fold.nunique().max() == 1
print('V1 train images:', len(manifest), '| near pairs:', len(near_pairs), '| dataset:', DATA_ROOT)

## 3. Teacher ConvNeXt-Tiny pretrained secara OOF

Setiap gambar diprediksi oleh model yang tidak pernah melatih fold gambar tersebut. Seluruh layer ConvNeXt-Tiny pretrained di-fine-tune pada train resmi; bobot pretrained hanya menjadi inisialisasi. Checkpoint dipilih memakai validation macro-F1.


In [ ]:
def load_rgb(path):
    with Image.open(path) as image:
        if image.mode == 'P':
            image = image.convert('RGBA')
        image = ImageOps.exif_transpose(image)
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            background = Image.new('RGB', rgba.size, tuple(int(v * 255) for v in IMAGENET_MEAN))
            background.paste(rgba, mask=rgba.getchannel('A'))
            return background
        return image.convert('RGB')


class Letterbox:
    def __init__(self, size):
        self.size = size
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), Image.Resampling.BILINEAR,
                            color=tuple(int(v * 255) for v in IMAGENET_MEAN))


teacher_train_transform = transforms.Compose([
    Letterbox(TEACHER_IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5, fill=tuple(int(v * 255) for v in IMAGENET_MEAN)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.08, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
teacher_eval_transform = transforms.Compose([
    Letterbox(TEACHER_IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class TeacherDataset(Dataset):
    def __init__(self, frame, training):
        self.frame = frame.reset_index(drop=True)
        self.transform = teacher_train_transform if training else teacher_eval_transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.transform(load_rgb(row.source_path)), CLASS_TO_INDEX[row.label_original], index


def teacher_loader(dataset, sampler=None, shuffle=False, batch_size=None):
    selected_batch = batch_size or TEACHER_BATCH_SIZE
    if selected_batch is None:
        raise RuntimeError('Physical batch belum dipilih oleh probe VRAM.')
    options = dict(dataset=dataset, batch_size=selected_batch, sampler=sampler,
                   shuffle=shuffle if sampler is None else False, num_workers=NUM_WORKERS,
                   pin_memory=True)
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


def teacher_sampler(frame):
    counts = frame.label_original.value_counts()
    weights = frame.label_original.map(lambda label: counts[label] ** -0.5).to_numpy(copy=True)
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), len(frame), replacement=True)


def teacher_model():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(CLASSES))
    return model.to(device='cuda', memory_format=torch.channels_last)


def probe_teacher_batch_size():
    if not torch.cuda.is_available():
        raise RuntimeError('Aktifkan GPU Colab untuk OOF teacher.')
    for batch_size in TEACHER_BATCH_CANDIDATES:
        if batch_size > TEACHER_TARGET_BATCH_SIZE or TEACHER_TARGET_BATCH_SIZE % batch_size:
            continue
        model = optimizer = images = targets = loss = None
        try:
            torch.cuda.empty_cache()
            model = teacher_model()
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=TEACHER_LR, weight_decay=TEACHER_WEIGHT_DECAY,
            )
            images = torch.zeros(
                (batch_size, 3, TEACHER_IMAGE_SIZE, TEACHER_IMAGE_SIZE),
                device='cuda',
            ).contiguous(memory_format=torch.channels_last)
            targets = torch.zeros(batch_size, dtype=torch.long, device='cuda')
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = nn.functional.cross_entropy(model(images), targets)
            loss.backward()
            optimizer.step()
            print(f'Physical batch {batch_size}: cocok.')
            return batch_size
        except RuntimeError as exc:
            if 'out of memory' not in str(exc).lower():
                raise
            print(f'Physical batch {batch_size}: OOM, mencoba batch lebih kecil.')
        finally:
            del model, optimizer, images, targets, loss
            torch.cuda.empty_cache()
    raise RuntimeError('Batch 16 tidak muat. Gunakan GPU dengan VRAM lebih besar atau turunkan resolusi.')


if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan GPU Colab untuk OOF teacher.')

TEACHER_BATCH_SIZE = probe_teacher_batch_size()
TEACHER_GRAD_ACCUM_STEPS = TEACHER_TARGET_BATCH_SIZE // TEACHER_BATCH_SIZE
print(
    'Teacher:', TEACHER_MODEL,
    '| input:', TEACHER_IMAGE_SIZE,
    '| physical batch:', TEACHER_BATCH_SIZE,
    '| accumulation:', TEACHER_GRAD_ACCUM_STEPS,
    '| effective batch:', TEACHER_BATCH_SIZE * TEACHER_GRAD_ACCUM_STEPS,
)


def train_teacher_fold(fold):
    fold_dir = OUTPUT_DIR / 'teacher_oof' / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = fold_dir / 'best.pt'
    complete_path = fold_dir / 'complete.json'
    train_frame = manifest[manifest.fold.ne(fold)].reset_index(drop=True)
    valid_frame = manifest[manifest.fold.eq(fold)].reset_index(drop=True)
    if checkpoint_path.is_file() and complete_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)
        model = teacher_model()
        model.load_state_dict(checkpoint['model_state'])
        print(f'Teacher fold {fold}: memakai checkpoint lengkap.')
        return model, valid_frame, checkpoint

    train_loader = teacher_loader(TeacherDataset(train_frame, True), teacher_sampler(train_frame))
    valid_loader = teacher_loader(TeacherDataset(valid_frame, False))
    model = teacher_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=TEACHER_LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=TEACHER_LR, weight_decay=TEACHER_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TEACHER_EPOCHS, eta_min=TEACHER_LR * 0.05,
    )
    scaler = torch.amp.GradScaler('cuda')
    best, stale = -1.0, 0

    for epoch in range(TEACHER_EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, (images, targets, _) in enumerate(
            tqdm(train_loader, desc=f'teacher fold {fold} epoch {epoch + 1}', leave=False)
        ):
            images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
            targets = targets.cuda(non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = criterion(model(images), targets) / TEACHER_GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            update_now = (step + 1) % TEACHER_GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader)
            if update_now:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), TEACHER_MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        model.eval()
        probabilities, targets_all, indices_all = [], [], []
        with torch.inference_mode():
            for images, targets, indices in valid_loader:
                images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
                with torch.autocast('cuda', dtype=torch.float16):
                    probs = model(images).softmax(1)
                probabilities.append(probs.float().cpu().numpy())
                targets_all.append(targets.numpy())
                indices_all.append(indices.numpy())
        probabilities = np.concatenate(probabilities)
        targets = np.concatenate(targets_all)
        order = np.argsort(np.concatenate(indices_all))
        score = f1_score(targets[order], probabilities[order].argmax(1), average='macro', zero_division=0)
        print(f'teacher fold {fold} epoch {epoch + 1}: macro_f1={score:.4f}')
        if score > best:
            best, stale = score, 0
            torch.save({
                'model_state': model.state_dict(),
                'epoch': epoch + 1,
                'macro_f1': score,
                'model': TEACHER_MODEL,
                'image_size': TEACHER_IMAGE_SIZE,
                'physical_batch_size': TEACHER_BATCH_SIZE,
                'effective_batch_size': TEACHER_TARGET_BATCH_SIZE,
            }, checkpoint_path)
        else:
            stale += 1
            if stale >= TEACHER_PATIENCE:
                break

    checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    complete_path.write_text(json.dumps({
        'fold': fold,
        'epoch': checkpoint['epoch'],
        'macro_f1': checkpoint['macro_f1'],
        'model': TEACHER_MODEL,
        'image_size': TEACHER_IMAGE_SIZE,
        'physical_batch_size': TEACHER_BATCH_SIZE,
        'gradient_accumulation_steps': TEACHER_GRAD_ACCUM_STEPS,
        'effective_batch_size': TEACHER_TARGET_BATCH_SIZE,
    }, indent=2))
    return model, valid_frame, checkpoint


oof_probabilities = np.full((len(manifest), len(CLASSES)), np.nan, dtype=np.float32)
teacher_metrics = []
for fold in range(FOLDS):
    model, valid_frame, checkpoint = train_teacher_fold(fold)
    loader = teacher_loader(TeacherDataset(valid_frame, False))
    probs_all, indices_all = [], []
    model.eval()
    with torch.inference_mode():
        for images, _, indices in loader:
            images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
            with torch.autocast('cuda', dtype=torch.float16):
                probs_all.append(model(images).softmax(1).float().cpu().numpy())
            indices_all.append(indices.numpy())
    probs = np.concatenate(probs_all)[np.argsort(np.concatenate(indices_all))]
    original_indices = manifest.index[manifest.fold.eq(fold)].to_numpy()
    oof_probabilities[original_indices] = probs
    teacher_metrics.append({
        'fold': fold,
        'best_epoch': checkpoint['epoch'],
        'macro_f1': checkpoint['macro_f1'],
        'physical_batch_size': TEACHER_BATCH_SIZE,
        'effective_batch_size': TEACHER_TARGET_BATCH_SIZE,
    })
    del model
    torch.cuda.empty_cache()

assert np.isfinite(oof_probabilities).all()
np.save(OUTPUT_DIR / 'teacher_oof_probabilities.npy', oof_probabilities)
pd.DataFrame(teacher_metrics).to_csv(OUTPUT_DIR / 'teacher_fold_metrics.csv', index=False)


## 4. Terapkan pseudo-label konservatif

Label mentah tidak ditimpa. Keputusan disimpan di `label_final` dan `label_decision`, beserta seluruh probabilitas OOF.


In [ ]:
original_indices = manifest.label_original.map(CLASS_TO_INDEX).to_numpy()
predicted_indices = oof_probabilities.argmax(1)
sorted_probabilities = np.sort(oof_probabilities, axis=1)
manifest['oof_predicted_label'] = [INDEX_TO_CLASS[index] for index in predicted_indices]
manifest['oof_original_probability'] = oof_probabilities[np.arange(len(manifest)), original_indices]
manifest['oof_alternative_probability'] = oof_probabilities.max(1)
manifest['oof_margin'] = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
for index, label in enumerate(CLASSES):
    manifest[f'oof_prob_{label}'] = oof_probabilities[:, index]

manifest['pseudo_candidate'] = (
    manifest.oof_predicted_label.ne(manifest.label_original)
    & manifest.oof_alternative_probability.ge(PSEUDO_MIN_ALTERNATIVE_PROB)
    & manifest.oof_original_probability.le(PSEUDO_MAX_ORIGINAL_PROB)
    & manifest.oof_margin.ge(PSEUDO_MIN_MARGIN)
)
manifest['label_final'] = manifest.label_original
manifest['label_decision'] = 'keep_original'

for original_label in CLASSES:
    candidates = manifest[
        manifest.pseudo_candidate & manifest.label_original.eq(original_label)
    ].sort_values(['oof_alternative_probability', 'oof_margin'], ascending=False)
    limit = max(1, int(math.floor(manifest.label_original.eq(original_label).sum() * MAX_RELABEL_FRACTION_PER_ORIGINAL_CLASS)))
    selected = candidates.head(limit).index
    manifest.loc[selected, 'label_final'] = manifest.loc[selected, 'oof_predicted_label']
    manifest.loc[selected, 'label_decision'] = 'oof_pseudo_relabel'

manifest.to_csv(OUTPUT_DIR / 'label_audit_before_dedup.csv', index=False)
print('Pseudo relabels:', int(manifest.label_decision.eq('oof_pseudo_relabel').sum()))
display(pd.crosstab(manifest.label_original, manifest.label_final).reindex(index=CLASSES, columns=CLASSES, fill_value=0))

## 5. Konsensus grup dan karantina

Near-duplicate dengan label final berbeda hanya diselesaikan bila seluruh prediksi OOF kuat dan sepakat. Jika tidak, seluruh grup dikarantina. Satu gambar teknis terbaik dipertahankan dari setiap grup yang terselesaikan.


In [ ]:
manifest['dedup_decision'] = 'keep'
manifest['dedup_reason'] = 'unique_or_group_canonical'
manifest['pixel_count'] = manifest.width * manifest.height
manifest['quality_penalty'] = (
    manifest.dark_candidate.astype(int) + manifest.bright_candidate.astype(int)
    + manifest.low_contrast_candidate.astype(int) + manifest.narrow_dynamic_range_candidate.astype(int)
)

for group_name, group in manifest.groupby('split_group'):
    if len(group) == 1:
        continue
    labels = set(group.label_final)
    working = group.copy()
    if len(labels) > 1:
        confident = working.oof_alternative_probability.ge(PSEUDO_MIN_ALTERNATIVE_PROB)
        predicted_labels = set(working.loc[confident, 'oof_predicted_label'])
        if confident.all() and len(predicted_labels) == 1:
            consensus = next(iter(predicted_labels))
            manifest.loc[working.index, 'label_final'] = consensus
            manifest.loc[working.index, 'label_decision'] = 'near_group_oof_consensus'
        else:
            manifest.loc[working.index, 'dedup_decision'] = 'exclude'
            manifest.loc[working.index, 'dedup_reason'] = 'unresolved_cross_label_near_group'
            continue
    working = manifest.loc[working.index].sort_values(
        ['quality_penalty', 'pixel_count', 'filepath'], ascending=[True, False, True], kind='stable'
    )
    duplicate_indices = working.index[1:]
    manifest.loc[duplicate_indices, 'dedup_decision'] = 'exclude'
    manifest.loc[duplicate_indices, 'dedup_reason'] = 'near_duplicate_noncanonical'

clean_manifest = manifest[manifest.dedup_decision.eq('keep')].copy().reset_index(drop=True)
excluded_near = manifest[manifest.dedup_decision.eq('exclude')].copy().reset_index(drop=True)
assert clean_manifest.fold.between(0, FOLDS - 1).all()
assert not clean_manifest.pixel_sha256.duplicated().any()
clean_manifest.to_csv(OUTPUT_DIR / 'clean_manifest_before_export.csv', index=False)
excluded_near.to_csv(OUTPUT_DIR / 'excluded_near_duplicates.csv', index=False)
print('After near dedup:', len(clean_manifest), '| removed:', len(excluded_near))
display(pd.crosstab(clean_manifest.fold, clean_manifest.label_final).reindex(columns=CLASSES, fill_value=0))

## 6. Audit visual hasil relabeling

Bagian ini menyimpan semua label otomatis yang benar-benar berubah ke `relabel_review_queue.csv`, lalu membuat contact sheet per halaman. Setiap tile menampilkan label asli, label ConvNeXt, confidence, margin, dan alasan keputusan.

Kolom `manual_verdict`, `manual_label`, dan `review_notes` disediakan sebagai catatan audit. Sesuai aturan kompetisi yang melarang manual labeling, isian manual **tidak digabung otomatis** ke manifest student training. Student hanya memakai keputusan otomatis OOF yang lolos threshold dan konsensus grup.


In [ ]:
REVIEW_DIR = OUTPUT_DIR / 'relabel_review'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_PAGE_SIZE = 16
REVIEW_COLUMNS = 4
REVIEW_THUMB_SIZE = (280, 210)

review_queue = clean_manifest[
    clean_manifest.label_final.ne(clean_manifest.label_original)
].copy()
review_queue['transition'] = review_queue.label_original + ' -> ' + review_queue.label_final
review_queue = review_queue.sort_values(
    ['transition', 'oof_alternative_probability', 'oof_margin'],
    ascending=[True, False, False],
).reset_index(drop=True)
review_queue.insert(0, 'review_id', [f'RELABEL-{index + 1:04d}' for index in range(len(review_queue))])
review_queue['manual_verdict'] = ''  # agree_with_model | keep_original | unsure
review_queue['manual_label'] = ''    # catatan audit saja; tidak masuk manifest training otomatis
review_queue['review_notes'] = ''

review_columns = [
    'review_id', 'filepath', 'source_path', 'label_original', 'label_final', 'transition',
    'label_decision', 'oof_original_probability', 'oof_alternative_probability',
    'oof_margin', 'oof_prob_Aman', 'oof_prob_Rusak', 'oof_prob_Lainnya',
    'fold', 'split_group', 'pixel_sha256',
    'manual_verdict', 'manual_label', 'review_notes',
]
review_columns = [column for column in review_columns if column in review_queue.columns]
review_queue[review_columns].to_csv(REVIEW_DIR / 'relabel_review_queue.csv', index=False)

transition_summary = (
    review_queue.groupby(['label_original', 'label_final', 'label_decision'], dropna=False)
    .size().rename('count').reset_index()
)
transition_summary.to_csv(REVIEW_DIR / 'relabel_transition_summary.csv', index=False)


def review_contact_sheet(frame, page_number):
    tile_width, image_height = REVIEW_THUMB_SIZE
    caption_height = 96
    tile_height = image_height + caption_height
    rows = math.ceil(len(frame) / REVIEW_COLUMNS)
    canvas = Image.new('RGB', (REVIEW_COLUMNS * tile_width, rows * tile_height), 'white')
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default()

    for position, (_, row) in enumerate(frame.iterrows()):
        column = position % REVIEW_COLUMNS
        grid_row = position // REVIEW_COLUMNS
        left, top = column * tile_width, grid_row * tile_height
        image = load_rgb(row.source_path)
        image = ImageOps.contain(image, (tile_width - 12, image_height - 12), Image.Resampling.LANCZOS)
        image_left = left + (tile_width - image.width) // 2
        image_top = top + (image_height - image.height) // 2
        canvas.paste(image, (image_left, image_top))
        draw.rectangle((left, top, left + tile_width - 1, top + tile_height - 1), outline='#bbbbbb', width=1)
        caption = [
            f"{row.review_id} | {row.label_original} -> {row.label_final}",
            f"confidence={row.oof_alternative_probability:.3f} | margin={row.oof_margin:.3f}",
            f"decision={row.label_decision}",
            Path(row.filepath).name,
        ]
        for line_number, line in enumerate(caption):
            draw.text((left + 8, top + image_height + 5 + line_number * 20), line, fill='black', font=font)

    output_path = REVIEW_DIR / f'relabel_contact_sheet_{page_number:03d}.jpg'
    canvas.save(output_path, quality=92, optimize=True)
    return output_path


contact_sheet_paths = []
for start in range(0, len(review_queue), REVIEW_PAGE_SIZE):
    page = review_queue.iloc[start:start + REVIEW_PAGE_SIZE]
    contact_sheet_paths.append(review_contact_sheet(page, len(contact_sheet_paths) + 1))

print('Relabel otomatis untuk direview:', len(review_queue))
display(transition_summary)
print('Review CSV:', REVIEW_DIR / 'relabel_review_queue.csv')
print('Contact sheets:', len(contact_sheet_paths))
if contact_sheet_paths:
    display(Image.open(contact_sheet_paths[0]))
else:
    print('Tidak ada label yang berubah; threshold konservatif menahan semua kandidat.')


## 7. Simpan label otomatis dan manifest student training

Student training berikutnya membaca `student_training_manifest.csv` dan gambar training mentah. Tidak ada gambar hasil crop atau hasil koreksi pixel yang dimaterialisasi.


In [ ]:
clean_manifest['label'] = clean_manifest.label_final
clean_manifest['label_source'] = np.where(
    clean_manifest.label_final.eq(clean_manifest.label_original),
    'original', 'convnext_oof_auto',
)
clean_manifest['source_split'] = 'train'
clean_manifest['use_for_training'] = True

output_columns = [
    'filepath', 'source_path', 'label', 'label_original', 'label_final', 'label_decision',
    'fold', 'split_group', 'pixel_sha256', 'oof_predicted_label',
    'oof_original_probability', 'oof_alternative_probability', 'oof_margin',
    'oof_prob_Aman', 'oof_prob_Rusak', 'oof_prob_Lainnya',
    'dark_candidate', 'bright_candidate', 'low_contrast_candidate',
    'narrow_dynamic_range_candidate', 'dedup_decision', 'dedup_reason',
    'label_source', 'source_split', 'use_for_training',
]
output_columns = [column for column in output_columns if column in clean_manifest.columns]
student_manifest = clean_manifest[output_columns].copy()
student_manifest.to_csv(OUTPUT_DIR / 'student_training_manifest.csv', index=False)
# Nama kompatibel untuk pipeline lama. Isinya identik dengan manifest student.
student_manifest.to_csv(OUTPUT_DIR / 'repaired_training_manifest.csv', index=False)

label_changes = manifest[manifest.label_decision.ne('keep_original')].copy()
label_changes.to_csv(OUTPUT_DIR / 'label_changes.csv', index=False)
excluded_near.to_csv(OUTPUT_DIR / 'quarantined_training_images.csv', index=False)

summary = {
    'policy': 'train_only_oof_relabeling',
    'teacher': 'ConvNeXt-Tiny ImageNet pretrained, fully fine-tuned on official train',
    'teacher_image_size': TEACHER_IMAGE_SIZE,
    'teacher_physical_batch_size': TEACHER_BATCH_SIZE,
    'teacher_gradient_accumulation_steps': TEACHER_GRAD_ACCUM_STEPS,
    'teacher_effective_batch_size': TEACHER_TARGET_BATCH_SIZE,
    'input_images': int(len(manifest)),
    'training_images_final': int(len(clean_manifest)),
    'pseudo_relabels': int(clean_manifest.label_decision.eq('oof_pseudo_relabel').sum()),
    'group_consensus_relabels': int(clean_manifest.label_decision.eq('near_group_oof_consensus').sum()),
    'quarantined_images': int(len(excluded_near)),
    'review_queue_rows': int(len(review_queue)),
    'review_contact_sheets': int(len(contact_sheet_paths)),
    'student_manifest': 'student_training_manifest.csv',
    'manual_review_applied_to_training': False,
    'class_counts_final': {label: int(clean_manifest.label.eq(label).sum()) for label in CLASSES},
    'test_pixels_read': False,
    'raw_files_modified': False,
}
(OUTPUT_DIR / 'relabel_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
archive_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)

assert clean_manifest.pixel_sha256.is_unique
assert clean_manifest.fold.between(0, FOLDS - 1).all()
assert set(clean_manifest.label) == set(CLASSES)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Student manifest:', OUTPUT_DIR / 'student_training_manifest.csv')
print('Review queue:', REVIEW_DIR / 'relabel_review_queue.csv')
print('ZIP:', archive_path)
